# 03 — Understanding Vibration Signals and Feature Extraction

## Module 3: what does the sensor give us?

A vibration sensor records how the bearing or test rig moves while it is operating. The computer receives a sequence of numerical readings, not a picture of the bearing.

Each reading is one **sample**. The value of that reading is the signal's **amplitude** at that instant. We can plot the values with time along the horizontal axis and amplitude along the vertical axis.

For this Paderborn workflow, the code uses a nominal sampling rate of 64,000 samples per second (64 kHz). That means the time gap represented by one sample is:

`1 / 64,000 = 0.000015625 seconds = 15.625 microseconds`

The notebook first loads one example file, identifies the named `vibration_1` channel, and extracts its numerical `Data` array. The plotting cells below show a short part of that real extracted recording so that the waveform is readable.

## What to look for in the waveform

- **Time (horizontal axis):** when each sensor sample was recorded.
- **Amplitude (vertical axis):** the recorded vibration value at that moment. The project file does not define a physical unit in this notebook, so do not label it as acceleration in `m/s²`.
- **Larger excursions:** larger positive or negative readings. They describe the signal amplitude; by themselves they do not prove which fault is present.
- **Faster oscillations:** more cycles in the same time interval, which means a higher frequency.

A real bearing signal is usually a mixture of many vibration components, so it can look irregular rather than like one clean sine wave.

## Frequency — an intuitive idea

Frequency describes how many times a repeating pattern completes a cycle in one second. Its unit is Hertz (Hz). For example, 100 Hz means 100 cycles per second.

A time waveform shows **how the signal changes over time**. A frequency spectrum shows **which oscillation rates are present and how strong their components are**. The optional spectrum plot is an introduction; the feature-extraction function later in this notebook uses the FFT to calculate frequency-domain features.

The example notebook output for `N09_M07_F10_K001_1.mat` contains 256,823 vibration samples. At the nominal 64 kHz rate this corresponds to about 4.013 seconds. The original output reports a minimum of -4.8798, a maximum of 4.6906, a mean of -0.01545 and a standard deviation of 0.3673 (in the signal's recorded units).

## Scope of this notebook

This notebook initially extracts 26 general time- and frequency-domain features from each recording. Later V2 experiments removed four redundant features and retained 22 for sequence modelling. The separately packaged binary baseline uses a different 34-feature schema; do not confuse the two pipelines.

Raw Paderborn MAT recordings are not included in Git. Place them under `data/raw/` and run Jupyter with its working directory set to the `notebooks` folder.

In [ ]:
# ============================================================
# FEATURE EXTRACTION SETUP
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Use one known measurement file for the initial test
sample_file = next(RAW_DIR.glob("K001/*.mat"))

print("Sample file:")
print(sample_file)

In [ ]:
# ============================================================
# INSPECT ONE MATLAB FILE
# ============================================================

mat = sio.loadmat(
    sample_file,
    squeeze_me=False,
    struct_as_record=False
)

print("Top-level variables:")
print(
    [
        key for key in mat.keys()
        if not key.startswith("__")
    ]
)

In [ ]:
# ============================================================
# INSPECT MATLAB STRUCTURE AFTER UNWRAPPING
# ============================================================

data_key = [
    key for key in mat.keys()
    if not key.startswith("__")
][0]

# Unwrap the (1, 1) NumPy array
data = mat[data_key][0, 0]

print("Data key:", data_key)
print("Data type:", type(data))

print("\nAvailable fields:")
for field in data._fieldnames:
    print("-", field)

In [ ]:
# ============================================================
# INSPECT X AND Y STRUCTURES
# ============================================================

X = data.X
Y = data.Y

print("X type:", type(X))
print("Y type:", type(Y))

print("\nX fields:")
if hasattr(X, "_fieldnames"):
    for field in X._fieldnames:
        print("-", field)

print("\nY fields:")
if hasattr(Y, "_fieldnames"):
    for field in Y._fieldnames:
        print("-", field)

In [ ]:
# ============================================================
# INSPECT ALL Y CHANNELS
# ============================================================

Y_channels = np.asarray(data.Y).flatten()

print("Number of Y channels:", len(Y_channels))
print("Y array shape:", np.asarray(data.Y).shape)

print("\nAvailable Y channels:")

for i, channel in enumerate(Y_channels):

    name = channel.Name

    # Convert MATLAB string array to Python string
    if isinstance(name, np.ndarray):
        name = str(name.flatten()[0])

    data_shape = np.asarray(channel.Data).shape

    print(
        f"{i}: {name} | Data shape: {data_shape}"
    )

In [ ]:
# ============================================================
# EXTRACT VIBRATION_1 ROBUSTLY
# ============================================================

def get_vibration_signal(mat_file):
    """
    Load one Paderborn .mat file and return the vibration_1 signal.
    """

    mat = sio.loadmat(
        mat_file,
        squeeze_me=False,
        struct_as_record=False
    )

    # Find the actual measurement variable
    data_keys = [
        key for key in mat.keys()
        if not key.startswith("__")
    ]

    if len(data_keys) != 1:
        raise ValueError(
            f"Unexpected MATLAB structure in {mat_file}"
        )

    data = mat[data_keys[0]][0, 0]

    # Find vibration channel by name
    y_channels = np.asarray(data.Y).flatten()

    for channel in y_channels:

        name = channel.Name

        if isinstance(name, np.ndarray):
            name = str(name.flatten()[0])

        if name == "vibration_1":

            signal = np.asarray(
                channel.Data
            ).squeeze()

            return signal.astype(np.float64)

    raise ValueError(
        f"vibration_1 channel not found in {mat_file}"
    )

In [ ]:
# ============================================================
# TEST VIBRATION EXTRACTION
# ============================================================

test_signal = get_vibration_signal(sample_file)

print("Signal extracted successfully")
print("Number of samples:", len(test_signal))
print("Minimum:", test_signal.min())
print("Maximum:", test_signal.max())
print("Mean:", test_signal.mean())
print("Standard deviation:", test_signal.std())

## Plot the vibration waveform

This is a short view of the real signal extracted above. We display only the first 5 milliseconds so individual oscillations are easier to see. This is **only a plotting window**: the original `test_signal` remains unchanged and the feature-extraction function later receives the full recording.

In [ ]:
import matplotlib.pyplot as plt

PLOT_FS = 64000  # nominal Paderborn sampling rate used in this project
DISPLAY_MS = 5.0

number_to_plot = min(
    len(test_signal),
    int(PLOT_FS * DISPLAY_MS / 1000)
)

time_ms = np.arange(number_to_plot) * 1000 / PLOT_FS

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(time_ms, test_signal[:number_to_plot], linewidth=0.7)
ax.set_title("Paderborn vibration waveform — first 5 milliseconds")
ax.set_xlabel("Time (milliseconds)")
ax.set_ylabel("Vibration amplitude (recorded units)")
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## A first look at frequency content

A waveform is drawn against time. A frequency spectrum is drawn against frequency. The next cell applies the same basic real-valued FFT operation that appears later in the feature-extraction function. For now, read it as a visual aid: peaks indicate frequencies with stronger spectral magnitude in this recording.

The largest peak is called the **dominant frequency** in this feature pipeline. It is not automatically the bearing's BPFO/BPFI fault frequency and it does not by itself identify the fault class.

In [ ]:
# Remove the mean before viewing the spectrum, as in the feature-extraction function.
centered_signal = test_signal - np.mean(test_signal)

fft_values = np.fft.rfft(centered_signal)
frequencies = np.fft.rfftfreq(
    len(centered_signal),
    d=1 / PLOT_FS
)
magnitudes = np.abs(fft_values)

# Limit the display to 0–20 kHz for readability.
visible = (frequencies > 0) & (frequencies <= 20000)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(frequencies[visible], magnitudes[visible], linewidth=0.7)
ax.set_title("Magnitude spectrum of the same vibration recording")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("FFT magnitude")
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## Module 4 — The first three time-domain features

We now begin turning the raw vibration samples into a small set of descriptive numbers. The first three values in the code are **mean**, **standard deviation** and **RMS**.

### 1. Mean — the average value

The mean is the ordinary arithmetic average of all signal samples:

mean = (x₁ + x₂ + … + xₙ) / n

It describes the signal's average level or offset. A mean close to zero says the samples average out around zero; it does **not** say that the vibration itself is small.

In the implementation:

`mean_value = np.mean(signal)`

### 2. Standard deviation — how spread out are the samples?

Standard deviation measures how far the readings typically spread around their mean. In this notebook, `np.std(signal)` uses NumPy's default `ddof=0` (population standard deviation):

std = sqrt(mean((x - mean)²))

A low standard deviation means most samples are close to the mean. A higher value means they are more spread out.

In the implementation:

`std_value = np.std(signal)`

### 3. RMS — the overall signal magnitude

RMS stands for **Root Mean Square**. We square each reading, average the squared readings, and then take the square root:

RMS = sqrt(mean(x²))

Squaring makes negative and positive readings contribute positively. RMS is a measure of the signal's overall magnitude relative to zero; unlike standard deviation, it also reflects a non-zero mean/offset.

In the implementation:

`rms_value = np.sqrt(np.mean(signal ** 2))`

### A tiny hand-worked example

For the simple signal `[1, 2, 3]`:

- Mean = `(1 + 2 + 3) / 3 = 2`
- Standard deviation = `sqrt(((1−2)² + (2−2)² + (3−2)²) / 3) = sqrt(2/3) ≈ 0.8165`
- RMS = `sqrt((1² + 2² + 3²) / 3) = sqrt(14/3) ≈ 2.1602`

They answer different questions: mean tells us the average level, standard deviation tells us the spread around that level, and RMS tells us the overall magnitude relative to zero.

The next small code cell calculates all three values from this toy signal so you can compare the formulas with NumPy.

In [ ]:
toy_signal = np.array([1.0, 2.0, 3.0])

toy_mean = np.mean(toy_signal)
toy_std = np.std(toy_signal)  # ddof=0 by default
toy_rms = np.sqrt(np.mean(toy_signal ** 2))

print(f"Toy signal: {toy_signal}")
print(f"Mean: {toy_mean:.4f}")
print(f"Standard deviation: {toy_std:.4f}")
print(f"RMS: {toy_rms:.4f}")

In [ ]:
# ============================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================

from scipy.stats import kurtosis, skew
import numpy as np


FS = 64000  # Paderborn vibration sampling frequency


def extract_vibration_features(signal, fs=FS):
    """
    Extract time-domain and frequency-domain features
    from one vibration signal.
    """

    signal = np.asarray(signal, dtype=np.float64).flatten()

    if len(signal) == 0:
        raise ValueError("Signal is empty.")

    # --------------------------------------------------------
    # Time-domain features
    # --------------------------------------------------------

    # Mean: average level of the raw vibration samples.
    mean_value = np.mean(signal)

    # Standard deviation: spread around the mean.
    # NumPy's default ddof=0 gives population standard deviation.
    std_value = np.std(signal)

    # RMS: overall magnitude relative to zero.
    rms_value = np.sqrt(
        np.mean(signal ** 2)
    )

    min_value = np.min(signal)
    max_value = np.max(signal)

    peak_to_peak = max_value - min_value

    kurtosis_value = kurtosis(
        signal,
        fisher=True,
        bias=False
    )

    skewness_value = skew(
        signal,
        bias=False
    )

    crest_factor = (
        np.max(np.abs(signal)) / rms_value
        if rms_value != 0 else 0
    )

    # --------------------------------------------------------
    # Frequency-domain features
    # --------------------------------------------------------

    n = len(signal)

    # Remove DC component before spectral analysis
    centered_signal = signal - mean_value

    fft_values = np.fft.rfft(centered_signal)
    frequencies = np.fft.rfftfreq(
        n,
        d=1 / fs
    )

    magnitude = np.abs(fft_values)

    # Ignore DC bin when finding dominant frequency
    magnitude_no_dc = magnitude.copy()

    if len(magnitude_no_dc) > 1:
        magnitude_no_dc[0] = 0

    dominant_index = np.argmax(
        magnitude_no_dc
    )

    dominant_frequency = frequencies[
        dominant_index
    ]

    dominant_magnitude = magnitude[
        dominant_index
    ]

    # Power spectrum
    power = magnitude ** 2

    total_power = np.sum(power)

    if total_power > 0:

        spectral_centroid = (
            np.sum(frequencies * power)
            / total_power
        )

        spectral_bandwidth = np.sqrt(
            np.sum(
                ((frequencies - spectral_centroid) ** 2)
                * power
            )
            / total_power
        )

        probability = power / total_power

        spectral_entropy = -np.sum(
            probability[
                probability > 0
            ]
            * np.log2(
                probability[
                    probability > 0
                ]
            )
        )

    else:

        spectral_centroid = 0.0
        spectral_bandwidth = 0.0
        spectral_entropy = 0.0

    spectral_rms = np.sqrt(
        np.mean(power)
    )

    # --------------------------------------------------------
    # Frequency band powers
    # --------------------------------------------------------

    bands = [
        (0, 500),
        (500, 1000),
        (1000, 2000),
        (2000, 3000),
        (3000, 4000),
        (4000, 5000),
        (5000, 6000),
        (6000, 7000),
        (7000, 8000),
        (8000, 9000),
        (9000, 10000)
    ]

    features = {
        "mean": mean_value,
        "std": std_value,
        "rms": rms_value,
        "min": min_value,
        "max": max_value,
        "peak_to_peak": peak_to_peak,
        "kurtosis": kurtosis_value,
        "skewness": skewness_value,
        "crest_factor": crest_factor,
        "dominant_frequency": dominant_frequency,
        "dominant_magnitude": dominant_magnitude,
        "spectral_centroid": spectral_centroid,
        "spectral_bandwidth": spectral_bandwidth,
        "spectral_rms": spectral_rms,
        "spectral_entropy": spectral_entropy
    }

    for low, high in bands:

        mask = (
            (frequencies >= low)
            & (frequencies < high)
        )

        features[
            f"band_power_{low}_{high}"
        ] = np.sum(
            power[mask]
        )

    return features

In [ ]:
# ============================================================
# TEST FEATURE EXTRACTION ON ONE SIGNAL
# ============================================================

sample_features = extract_vibration_features(
    test_signal
)

feature_test_df = pd.DataFrame(
    [sample_features]
)

print(
    "Number of extracted features:",
    len(sample_features)
)

display(feature_test_df.T)

## Read the three values from the real Paderborn recording

The original example run for `N09_M07_F10_K001_1.mat` reported approximately:

| Feature | Original notebook value |
|---|---:|
| Mean | -0.015454 |
| Standard deviation | 0.367295 |
| RMS | 0.367620 |

The mean is close to zero, so standard deviation and RMS are also close. They are not the same measure: standard deviation describes spread around the mean, while RMS measures magnitude relative to zero.

These values describe this one example recording in its recorded signal units. They are not universal healthy-bearing thresholds and do not diagnose a bearing on their own.

In [ ]:
# ============================================================
# TEST REPRESENTATIVE FILE FROM EACH CATEGORY
# ============================================================

representative_bearings = {
    "healthy": "K001",
    "outer": "KA01",
    "inner": "KI01",
    "both": "KB23"
}

representative_results = []

for category, bearing in representative_bearings.items():

    file = next(
        (RAW_DIR / bearing).glob("*.mat")
    )

    signal = get_vibration_signal(file)

    features = extract_vibration_features(
        signal
    )

    features["bearing"] = bearing
    features["category"] = category
    features["file"] = file.name

    representative_results.append(
        features
    )

representative_df = pd.DataFrame(
    representative_results
)

print(
    "Representative files processed:",
    len(representative_df)
)

display(
    representative_df[
        ["bearing", "category", "file"]
    ]
)

In [ ]:
# ============================================================
# BATCH FEATURE EXTRACTION - DIAGNOSTIC VERSION
# ============================================================

from pathlib import Path
import time

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

metadata_path = PROCESSED_DIR / "bearing_metadata.csv"
bearing_metadata = pd.read_csv(metadata_path)

# ------------------------------------------------------------
# Collect all MATLAB files
# ------------------------------------------------------------

# Scan only the 32 Paderborn bearing folders.
# Supplementary datasets such as cwru_48k_de may also live under data/raw,
# but they must not be included in the Paderborn 2,560-file batch.
mat_files = sorted(
    mat_file
    for bearing in bearing_metadata["bearing"].astype(str)
    for mat_file in (RAW_DIR / bearing).glob("*.mat")
)

print("Total .mat files found:", len(mat_files))

if len(mat_files) != 2560:
    raise ValueError(
        f"Expected 2560 .mat files, found {len(mat_files)}"
    )

# ------------------------------------------------------------
# Extract features
# ------------------------------------------------------------

all_features = []
failed_files = []

start_time = time.time()

for i, mat_file in enumerate(mat_files, start=1):

    try:

        # Extract vibration signal
        signal = get_vibration_signal(mat_file)

        # Extract features
        features = extract_vibration_features(signal)

        # ----------------------------------------------------
        # Parse filename
        # ----------------------------------------------------

        parts = mat_file.stem.split("_")

        if len(parts) < 5:
            raise ValueError(
                f"Unexpected filename format: {mat_file.name}"
            )

        operating_condition = "_".join(parts[:3])
        bearing = parts[3]
        measurement_number = int(parts[4])

        # ----------------------------------------------------
        # Get metadata
        # ----------------------------------------------------

        metadata_row = bearing_metadata[
            bearing_metadata["bearing"] == bearing
        ]

        if len(metadata_row) != 1:
            raise ValueError(
                f"Metadata not found uniquely for {bearing}"
            )

        metadata_row = metadata_row.iloc[0]

        # ----------------------------------------------------
        # Add metadata to features
        # ----------------------------------------------------

        features["file"] = mat_file.name
        features["bearing"] = bearing
        features["operating_condition"] = operating_condition
        features["measurement_number"] = measurement_number

        features["fault_location"] = (
            metadata_row["fault_location"]
        )

        features["fault_class"] = int(
            metadata_row["fault_class"]
        )

        features["binary_label"] = int(
            metadata_row["binary_label"]
        )

        all_features.append(features)

    except Exception as e:

        failed_files.append({
            "file": mat_file.name,
            "path": str(mat_file),
            "error_type": type(e).__name__,
            "error": str(e)
        })

        print(
            f"\nFAILED FILE #{i}: {mat_file.name}"
        )
        print(
            f"Error: {type(e).__name__}: {e}"
        )

    # --------------------------------------------------------
    # Progress report
    # --------------------------------------------------------

    if i % 100 == 0 or i == len(mat_files):

        elapsed = time.time() - start_time

        print(
            f"Processed {i}/{len(mat_files)} files "
            f"({i / len(mat_files) * 100:.1f}%) "
            f"| successful: {len(all_features)} "
            f"| failed: {len(failed_files)} "
            f"| elapsed: {elapsed / 60:.1f} min"
        )

# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

feature_df = pd.DataFrame(all_features)
failed_df = pd.DataFrame(failed_files)

print("\n==============================================")
print("EXTRACTION FINISHED")
print("==============================================")

print("Successful files:", len(feature_df))
print("Failed files:", len(failed_df))

if len(failed_df) > 0:
    print("\nFailed files:")
    display(failed_df)


In [ ]:
# ============================================================
# SAVE SUCCESSFUL FEATURES AND FAILED-FILE LOG
# ============================================================

feature_output_path = (
    PROCESSED_DIR / "paderborn_vibration_features.csv"
)

failure_output_path = (
    PROCESSED_DIR / "failed_mat_files.csv"
)

# Save successfully processed files
feature_df.to_csv(
    feature_output_path,
    index=False
)

# Save failed-file provenance
failed_df.to_csv(
    failure_output_path,
    index=False
)

print("Successful feature rows:", len(feature_df))
print("Failed files:", len(failed_df))

print("\nFeature dataset saved:")
print(feature_output_path.resolve())

print("\nFailure log saved:")
print(failure_output_path.resolve())

In [ ]:
# ============================================================
# VERIFY FINAL FEATURE DATASET
# ============================================================

print("Total feature rows:", len(feature_df))
print("Unique bearings:", feature_df["bearing"].nunique())
print(
    "Unique operating conditions:",
    feature_df["operating_condition"].nunique()
)

print("\nRows per bearing:")
print(
    feature_df.groupby("bearing").size()
)

print("\nFault-location distribution:")
print(
    feature_df["fault_location"].value_counts()
)

print("\nMissing values:")
print(
    feature_df.isna().sum().sum()
)

print("\nFailed files:")
display(failed_df)